# 08 · Dashboard Prototype

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Objective

This notebook creates a lightweight Python dashboard prototype using pandas and matplotlib.

A full Streamlit app is also included in `dashboard/streamlit_app.py`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users["has_support_call"] = users["user_id"].isin(support["user_id"]).astype(int)
users["upload_now"] = users["upload_choice"].eq("upload_now").astype(int)

users.head()

## 2. Dashboard KPIs

In [ ]:
kpi_table = pd.DataFrame({
    "KPI": [
        "Total Users",
        "Upload-now Rate",
        "Document Submission Rate",
        "Approval Rate",
        "Support Contact Rate",
        "Average Minutes to Upload"
    ],
    "Value": [
        f"{len(users):,}",
        f"{users['upload_now'].mean():.1%}",
        f"{users['document_submitted'].mean():.1%}",
        f"{users['approved'].mean():.1%}",
        f"{users['has_support_call'].mean():.1%}",
        f"{users['minutes_to_upload'].mean():.1f}"
    ]
})
kpi_table

## 3. KPI Trend Dashboard

In [ ]:
monthly = users.groupby("signup_month").agg(
    users=("user_id", "count"),
    upload_now_rate=("upload_now", "mean"),
    approval_rate=("approved", "mean"),
    support_rate=("has_support_call", "mean")
)

ax = monthly[["upload_now_rate", "approval_rate", "support_rate"]].plot(figsize=(11, 5), marker="o")
ax.set_title("Monthly KPI Dashboard")
ax.set_ylabel("Rate")
ax.set_xlabel("Signup Month")
plt.tight_layout()
plt.show()

## 4. Segment Dashboard

In [ ]:
segment = users.groupby("channel").agg(
    users=("user_id", "count"),
    upload_now_rate=("upload_now", "mean"),
    approval_rate=("approved", "mean"),
    support_rate=("has_support_call", "mean")
).sort_values("approval_rate", ascending=False)

segment

In [ ]:
ax = segment[["upload_now_rate", "approval_rate", "support_rate"]].plot(kind="bar", figsize=(11, 5))
ax.set_title("Channel Performance Dashboard")
ax.set_ylabel("Rate")
ax.set_xlabel("Channel")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 5. Funnel Dashboard

In [ ]:
funnel = pd.DataFrame({
    "step": ["Started", "P1 Completed", "P2 Reached", "Document Submitted", "Approved"],
    "users": [
        len(users),
        users["p1_completed"].sum(),
        users["p2_reached"].sum(),
        users["document_submitted"].sum(),
        users["approved"].sum()
    ]
})
funnel["conversion_from_start"] = funnel["users"] / funnel.loc[0, "users"]
funnel

In [ ]:
ax = funnel.set_index("step")["conversion_from_start"].plot(kind="bar", figsize=(10, 5))
ax.set_title("Funnel Conversion from Start")
ax.set_ylabel("Conversion from start")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. Executive Dashboard Notes

This notebook version is useful for static analysis.

For a recruiter-friendly interactive experience, use:

```bash
streamlit run dashboard/streamlit_app.py
```